# Mental Health Groq + Quiet RAG Chatbot

Pipeline: load `../docs/*.md`, `../docs/*.txt`, and `../docs/*.pdf` -> chunk -> embed -> FAISS -> retrieve hidden context -> generate a natural Groq reply.

The user-facing answer should not show source names, file references, line references, or citation markers.

In [ ]:
import json, os, pickle, re
from pathlib import Path
from urllib import request as urlrequest
import numpy as np
import faiss

DOCS = Path('../docs')
STORE = Path('../models/faiss_store')
STORE.mkdir(parents=True, exist_ok=True)

## Load references

PDF support uses `pypdf`. If you only have markdown/text references, this cell still works without PDFs.

In [ ]:
def read_pdf(path):
    from pypdf import PdfReader
    return '\n'.join((page.extract_text() or '') for page in PdfReader(path).pages)

def load_docs():
    docs = []
    for path in sorted(DOCS.iterdir()):
        if path.suffix.lower() in {'.md', '.txt'}:
            text = path.read_text(encoding='utf-8')
        elif path.suffix.lower() == '.pdf':
            text = read_pdf(path)
        else:
            continue
        if text.strip():
            docs.append((path.name, text))
    return docs

docs = load_docs()
[(name, len(text)) for name, text in docs]

In [ ]:
def chunk(t, size=400, overlap=80):
    out, i = [], 0
    while i < len(t):
        out.append(t[i:i+size])
        i += size - overlap
    return [c.strip() for c in out if c.strip()]

chunks, doc_meta = [], []
for name, text in docs:
    doc_chunks = chunk(text)
    chunks.extend(doc_chunks)
    doc_meta.append({'name': name, 'chunks': len(doc_chunks)})
len(chunks), doc_meta[:3]

## Build embeddings and FAISS index

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embs = model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    backend = 'sbert'
except Exception:
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words='english', sublinear_tf=True)
    embs = vec.fit_transform(chunks).toarray().astype(np.float32)
    embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-9)
    pickle.dump(vec, open(STORE / 'tfidf_vec.pkl', 'wb'))
    backend = 'tfidf'

index = faiss.IndexFlatIP(embs.shape[1])
index.add(embs)
backend, embs.shape

In [ ]:
def retrieve(q, k=3):
    if backend == 'sbert':
        qv = model.encode([q], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    else:
        qv = vec.transform([q]).toarray().astype(np.float32)
        qv = qv / (np.linalg.norm(qv, axis=1, keepdims=True) + 1e-9)
    scores, ids = index.search(qv, k)
    return [chunks[i] for i in ids[0] if i >= 0]

retrieve('How can I calm down when I feel anxious?')[0][:250]

## Generate with Groq

Set `GROQ_API_KEY` in your environment before running this cell. The prompt explicitly hides retrieval details from the final answer.

In [ ]:
GROQ_MODEL = os.getenv('GROQ_MODEL', 'llama-3.1-8b-instant')

def clean_answer(text):
    text = re.sub(r'\s+', ' ', text or '').strip()
    text = re.sub(r'(?i)\b(source|reference|file|line|citation)s?\s*[:#-]?\s*\S*', '', text).strip()
    return re.sub(r'\[(?:\d+|source[^\]]*)\]', '', text, flags=re.I).strip()

def groq_answer(q):
    context = '\n\n'.join(retrieve(q, 3))
    payload = {
        'model': GROQ_MODEL,
        'messages': [
            {'role': 'system', 'content': 'You are a warm, normal mental-health support chatbot. Use reference context quietly when it helps, but never mention sources, PDFs, files, line numbers, retrieval, or citations. Keep replies short and conversational. Do not diagnose.'},
            {'role': 'user', 'content': f'Reference context:\n{context}\n\nUser message: {q}'}
        ],
        'temperature': 0.45,
        'max_tokens': 260,
    }
    req = urlrequest.Request(
        'https://api.groq.com/openai/v1/chat/completions',
        data=json.dumps(payload).encode('utf-8'),
        headers={'Authorization': f'Bearer {os.environ["GROQ_API_KEY"]}', 'Content-Type': 'application/json'},
        method='POST',
    )
    data = json.loads(urlrequest.urlopen(req, timeout=20).read().decode('utf-8'))
    return clean_answer(data['choices'][0]['message']['content'])

# groq_answer('I feel anxious today. What can I do right now?')

## Save artifacts for Flask

In [ ]:
faiss.write_index(index, str(STORE / 'index.faiss'))
json.dump(chunks, open(STORE / 'chunks.json', 'w', encoding='utf-8'))
json.dump({'backend': backend, 'dim': int(embs.shape[1]), 'documents': doc_meta}, open(STORE / 'meta.json', 'w', encoding='utf-8'))
print('saved.')